# Deteksi Spam Email Berbahasa Indonesia dengan Machine Learning
**Bidang AI:** Machine Learning (klasifikasi teks) &nbsp;|&nbsp; **Studi kasus:** Penyaring spam email layanan kampus (Sistem dan Teknologi Informasi)

Alur notebook: (1) siapkan data → (2) bersihkan teks → (3) ubah ke vektor TF-IDF → (4) latih 3 model → (5) evaluasi → (6) analisis kesalahan → (7) demo prediksi.

> Dataset default adalah **dataset sintetis buatan sendiri** (3.000 email). Untuk hasil yang lebih realistis, lihat sel opsional di bagian akhir yang memakai dataset publik.

## 1. Menyiapkan data
Sel berikut membuat berkas `data/emails_sintetis.csv` (jalan otomatis di Google Colab, tidak perlu unggah berkas).

In [ ]:
"""Membuat dataset email sintetis berbahasa Indonesia (spam vs ham) untuk demo.
Dataset dibuat dari templat + kalimat pengisi acak, dengan sengaja memuat kasus sulit
(email sah yang memakai kata 'gratis', 'diskon', 'verifikasi', 'segera') serta
sedikit noise label agar evaluasi model tidak terlalu ideal.
"""
import random, csv, os
random.seed(42)

nama = ["Andi","Budi","Citra","Dewi","Eka","Fajar","Gita","Hendra","Intan","Joko","Kirana","Lukman","Maya","Nadia","Oki","Putri","Rangga","Sari","Tono","Wulan"]
kota = ["Bandar Lampung","Jakarta","Bandung","Surabaya","Yogyakarta","Medan","Semarang","Makassar"]
bank = ["BCA","BRI","BNI","Mandiri","BSI","CIMB"]
matkul = ["Basis Data","Jaringan Komputer","Pemrograman Web","Sistem Operasi","Kecerdasan Buatan","Rekayasa Perangkat Lunak","Statistika","Keamanan Informasi"]
hari = ["Senin","Selasa","Rabu","Kamis","Jumat"]
layanan = ["WiFi kampus","portal akademik","email kampus","VPN","server lab","printer perpustakaan","aplikasi presensi"]
domain_spam = ["bit.ly/x9Kd2","tinyurl.com/promo88","klik-akun-aman.xyz","hadiah-nasional.top","cek-saldo-bank.online"]
angka = lambda a,b: random.randint(a,b)

filler_netral = [
 "Terima kasih atas perhatiannya.","Mohon dicek kembali informasi berikut.","Salam hormat.","Silakan hubungi kami bila ada pertanyaan.",
 "Informasi ini dikirim otomatis, mohon tidak membalas email ini.","Harap segera menindaklanjuti sesuai kebutuhan.",
 "Kami tunggu konfirmasi Anda.","Semoga sehat selalu.","Mohon maaf atas ketidaknyamanannya."]

def spam():
    t = random.choice(range(10))
    if t==0:
        s=f"PERINGATAN! Akun {random.choice(bank)} Anda akan diblokir dalam 24 jam. Segera verifikasi data di {random.choice(domain_spam)} untuk menghindari pemblokiran."
    elif t==1:
        s=f"Selamat! Anda terpilih sebagai pemenang undian hadiah Rp {angka(10,500)}.000.000. Klik {random.choice(domain_spam)} dan kirim data rekening untuk klaim hadiah gratis."
    elif t==2:
        s=f"Butuh dana cepat? Pinjaman online cair {angka(5,30)} menit tanpa jaminan, tanpa survei. Bunga rendah, daftar sekarang di {random.choice(domain_spam)}."
    elif t==3:
        s=f"Bonus deposit {angka(50,300)}% untuk member baru! Situs slot gacor resmi, menang mudah setiap hari. Daftar gratis di {random.choice(domain_spam)}."
    elif t==4:
        s=f"Investasi kripto untung {angka(10,90)}% per minggu, dijamin tanpa risiko. Transfer sekarang, kuota terbatas! Hubungi WhatsApp admin."
    elif t==5:
        s=f"Lowongan kerja online dari rumah, gaji Rp {angka(3,15)} juta per bulan tanpa pengalaman. Cukup bayar biaya pendaftaran dan kirim KTP Anda."
    elif t==6:
        s=f"Paket Anda tertahan di bea cukai. Bayar biaya administrasi Rp {angka(15,90)}.000 hari ini via {random.choice(domain_spam)} agar paket segera dikirim."
    elif t==7:
        s=f"Obat herbal ampuh, hasil dalam {angka(3,14)} hari, diskon {angka(50,80)}% hanya hari ini! Pesan sekarang, stok terbatas, gratis ongkir."
    elif t==8:
        s=f"Kami mendeteksi login mencurigakan pada akun Anda. Konfirmasi kata sandi Anda di {random.choice(domain_spam)} sekarang juga atau akun ditutup permanen."
    else:
        s=f"Dapatkan iPhone terbaru hanya Rp {angka(1,3)}00.000! Promo kilat, klik link berikut dan isi data diri Anda untuk mendapatkannya gratis."
    if random.random()<0.35:
        s=random.choice(filler_netral)+" "+s
    return s

def ham():
    t = random.choice(range(12))
    n = random.choice(nama)
    if t==0:
        s=f"Yth. mahasiswa, ujian tengah semester mata kuliah {random.choice(matkul)} dilaksanakan hari {random.choice(hari)} pukul {angka(7,14)}.00 di ruang {angka(101,320)}."
    elif t==1:
        s=f"Halo {n}, tiket layanan #{angka(1000,9999)} terkait {random.choice(layanan)} sudah kami terima dan sedang diproses oleh tim TI. Estimasi selesai {angka(1,3)} hari kerja."
    elif t==2:
        s=f"Dear {n}, mohon kumpulkan tugas {random.choice(matkul)} melalui portal akademik paling lambat hari {random.choice(hari)}. Terima kasih."
    elif t==3:
        s=f"Undangan rapat koordinasi program studi hari {random.choice(hari)} pukul {angka(8,15)}.00 di ruang rapat. Mohon konfirmasi kehadiran."
    elif t==4:
        s=f"Bagi mahasiswa yang belum melakukan verifikasi KRS, silakan segera verifikasi melalui portal akademik resmi sebelum batas waktu hari {random.choice(hari)}."
    elif t==5:
        s=f"Seminar gratis 'Pengantar Kecerdasan Buatan' terbuka untuk seluruh mahasiswa, {random.choice(hari)} pukul {angka(9,15)}.00. Pendaftaran gratis melalui himpunan mahasiswa."
    elif t==6:
        s=f"Toko buku kampus memberi diskon {angka(10,25)}% untuk buku teks {random.choice(matkul)} sampai akhir bulan. Tunjukkan kartu mahasiswa saat pembelian."
    elif t==7:
        s=f"Hai {n}, jadi kita belajar bareng untuk ujian {random.choice(matkul)} di perpustakaan hari {random.choice(hari)}? Kabari ya."
    elif t==8:
        s=f"Pemberitahuan pemeliharaan: {random.choice(layanan)} tidak dapat diakses pada {random.choice(hari)} pukul {angka(0,4)}.00 sampai {angka(5,7)}.00 WIB. Mohon maaf atas ketidaknyamanannya."
    elif t==9:
        s=f"Pembayaran UKT semester ini dapat dilakukan melalui bank {random.choice(bank)} resmi kampus sampai tanggal {angka(5,28)}. Bukti bayar diunggah di portal akademik."
    elif t==10:
        s=f"Notifikasi keamanan: kata sandi email kampus Anda berhasil diubah. Bila bukan Anda, segera hubungi helpdesk TI di gedung {random.choice('ABCD')} lantai {angka(1,3)}."
    else:
        s=f"Laporan praktikum {random.choice(matkul)} milik {n} sudah diterima. Nilai akan diumumkan di {random.choice(kota)} pada pertemuan berikutnya."
    if random.random()<0.35:
        s=s+" "+random.choice(filler_netral)
    return s

rows=[(spam(),"spam") for _ in range(1050)]+[(ham(),"ham") for _ in range(1950)]
random.shuffle(rows)
for i in range(int(0.02*len(rows))):      # noise label 2%
    t,l=rows[i]; rows[i]=(t,"ham" if l=="spam" else "spam")
random.shuffle(rows)

os.makedirs("data",exist_ok=True)
with open("data/emails_sintetis.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["teks","label"]); w.writerows(rows)
print("Selesai:",len(rows),"baris")

In [ ]:
import pandas as pd
df = pd.read_csv("data/emails_sintetis.csv")
print(df.shape)
print(df["label"].value_counts())
df.sample(5, random_state=1)

## 2. Pembersihan teks, pemisahan data, dan pelatihan model

In [ ]:
import re, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

def bersihkan(t):
    t = t.lower()
    t = re.sub(r"https?://\S+|\b\S+\.(com|xyz|top|online)\S*|bit\.ly/\S+", " tautan ", t)
    t = re.sub(r"\d+", " angka ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

df["bersih"] = df["teks"].apply(bersihkan)
y = (df["label"] == "spam").astype(int)
Xtr, Xte, ytr, yte, tr_idx, te_idx = train_test_split(
    df["bersih"], y, df.index, test_size=0.2, stratify=y, random_state=42)

def pipa(clf):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)), ("clf", clf)])

model = {"Naive Bayes": MultinomialNB(alpha=0.5),
         "Logistic Regression": LogisticRegression(max_iter=1000, C=3.0),
         "Linear SVM": LinearSVC(C=1.0)}
hasil, terlatih = {}, {}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
for nm, clf in model.items():
    p = pipa(clf).fit(Xtr, ytr); terlatih[nm] = p
    pr = p.predict(Xte)
    hasil[nm] = dict(accuracy=accuracy_score(yte, pr), precision=precision_score(yte, pr),
                     recall=recall_score(yte, pr), f1=f1_score(yte, pr),
                     cv_f1=cross_val_score(pipa(clf), Xtr, ytr, cv=skf, scoring="f1").mean())
pd.DataFrame(hasil).T.round(4)

## 3. Evaluasi model terbaik

In [ ]:
terbaik = max(hasil, key=lambda k: hasil[k]["f1"])
p = terlatih[terbaik]; pred = p.predict(Xte)
print("Model terbaik:", terbaik)
print(classification_report(yte, pred, target_names=["ham","spam"], digits=4))
ConfusionMatrixDisplay(confusion_matrix(yte, pred), display_labels=["Ham","Spam"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix"); plt.show()

## 4. Kata penciri dan analisis kesalahan

In [ ]:
feat = p.named_steps["tfidf"].get_feature_names_out()
clf = p.named_steps["clf"]
w = clf.coef_[0] if hasattr(clf, "coef_") else clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
print("Penciri SPAM:", [feat[i] for i in np.argsort(w)[-10:][::-1]])
print("Penciri HAM :", [feat[i] for i in np.argsort(w)[:10]])
print()
salah = te_idx[pred != yte.values]
df.loc[salah, ["teks","label"]]

## 5. Demo: prediksi email baru

In [ ]:
contoh = ["Segera verifikasi akun bank Anda di bit.ly/abc123 atau akun diblokir hari ini!",
          "Yth. mahasiswa, jadwal ujian akhir Kecerdasan Buatan sudah tersedia di portal akademik.",
          "Selamat Anda menang undian Rp 50.000.000, kirim data rekening untuk klaim hadiah.",
          "Tiket layanan #4521 terkait WiFi kampus sedang diproses oleh tim TI.",
          "Seminar gratis keamanan siber untuk mahasiswa, daftar melalui himpunan."]
for c in contoh:
    label = p.predict([bersihkan(c)])[0]
    print("SPAM" if label else "HAM ", "|", c)

## 6. (Opsional) Coba dengan dataset publik
Sel di bawah memakai *SMS Spam Collection* (UCI). Dataset ini berbahasa Inggris dan berupa SMS, bukan email, tetapi cocok untuk membandingkan hasil pada data nyata. Butuh koneksi internet (Google Colab). Sel ini tidak diuji di lingkungan penyusun karena tanpa internet; bila URL berubah, unduh manual dari UCI Machine Learning Repository.

In [ ]:
# import urllib.request, zipfile, io
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
# z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
# sms = pd.read_csv(z.open("SMSSpamCollection"), sep="\t", names=["label","teks"])
# sms["bersih"] = sms["teks"].str.lower().str.replace(r"[^a-z\s]", " ", regex=True)
# y2 = (sms["label"]=="spam").astype(int)
# a,b,c,d = train_test_split(sms["bersih"], y2, test_size=0.2, stratify=y2, random_state=42)
# m2 = pipa(MultinomialNB(alpha=0.5)).fit(a,c)
# print(classification_report(d, m2.predict(b), target_names=["ham","spam"]))